# Notebook 3: Survival, pathway enrichment and sensitivity to panel size and hyperparameters
**Paper:** SHAP-Guided Selection of Gene Panel with Repeated Nested Cross-Validation and PAM50-Exclusion Benchmarking for Interpretable Breast Cancer Subtype Classification from RNA-Seq, with External Validation on METABRIC
**Authors:** Neetu Ahlawat, Vishal Verma

| Section | Analysis | Paper |
|---|---|---|
| 7 | Kaplan–Meier overall survival and log-rank tests of predicted subtypes in METABRIC | Section 4.8, Fig. 11 |
| 8 | Pathway enrichment of the SHAP-300 panel (Enrichr via GSEApy; needs internet) | Section 4.8 |
| 6 | Nested CV (first repeat, five outer folds): panel-size sweep (50–1,000 genes), inner-loop tuning of SVM and logistic regression, tests on balanced accuracy and macro F1 | Section 4.9, Fig. 12 |
| 6b | Tables and the panel-size figure for Section 6 | Fig. 12 |

**How to run:** *Runtime → Change runtime type → T4 GPU* (CPU also works, more slowly). Upload `HiSeqV2`, `TCGA.BRCA.sampleMap_BRCA_clinicalMatrix` and `brca_metabric.tar.gz` to `/content/`, then run all cells in order. Expected time: about 30 min with GPU, about 2 h with CPU. Results and figures are saved to `OUT_DIR`.


In [ ]:
# ===== 0. Setup and config =====
import os, sys, json, time, glob, tarfile, shutil, subprocess, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import binomtest
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split, GridSearchCV
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score,
                             precision_recall_fscore_support)
from sklearn.utils.class_weight import compute_sample_weight
import xgboost as xgb
from xgboost import XGBClassifier
warnings.filterwarnings("ignore")
try:
    display
except NameError:
    display = print

CONTENT      = os.environ.get("REV_CONTENT", "/content")
TCGA_EXPR    = f"{CONTENT}/HiSeqV2"                                   # .gz also works
TCGA_CLIN    = f"{CONTENT}/TCGA.BRCA.sampleMap_BRCA_clinicalMatrix"   # or BRCA_clinicalMatrix
METABRIC_DIR = f"{CONTENT}/brca_metabric"
OUT_DIR      = f"{CONTENT}/revision_results"
# To keep results if Colab disconnects, mount Drive and use a Drive folder instead:
# from google.colab import drive; drive.mount('/content/drive'); OUT_DIR = "/content/drive/MyDrive/revision_results"

SEED       = 42
PANEL_SIZE = int(os.environ.get("REV_K", 300))
K_LIST     = [int(k) for k in os.environ.get("REV_KLIST", "50,100,200,300,500,1000").split(",")]
N_REPEATS  = int(os.environ.get("REV_REPEATS", 5))   # 5 repeats x 5 folds = 25 outer folds, as in the paper
N_BOOT     = 1000
RUN_NOPAM50_IN_CV   = True    # SHAP-300 without PAM50 inside every fold (one extra booster per fold)
RUN_ALLGENE_XGB_CV  = True    # XGBoost on all 20,530 genes inside every fold (slow on CPU)

def gpu_available():
    try:
        return subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
    except Exception:
        return False
USE_GPU_CV = gpu_available()  # GPU is used only in Section 6; Sections 2-3 use CPU to match the earlier runs exactly
print("xgboost", xgb.__version__, "| GPU for Section 6:", USE_GPU_CV)

CLASSES     = ["Basal", "Her2", "LumA", "LumB"]
CLASS_NAMES = ["Basal-like", "HER2-enriched", "Luminal A", "Luminal B"]
LABEL_MAP   = {c: i for i, c in enumerate(CLASSES)}
os.makedirs(OUT_DIR, exist_ok=True)
SUMMARY = {}   # every key number is collected here and printed in Section 9

In [ ]:
# ===== Helper functions (same settings as Table 2 of the paper) =====
ALIAS = {"GSG2": "HASPIN", "ORC6L": "ORC6", "KNTC2": "NDC80", "CDCA1": "NUF2", "KIAA0101": "PCLAF",
         "FAM64A": "PIMREG", "HN1": "JPT1", "KIAA1199": "CEMIP", "C1orf106": "INAVA", "MLF1IP": "CENPU",
         "CTSL2": "CTSV", "C17orf37": "MIEN1", "C10orf116": "ADIRF"}
PAM50 = ["ACTR3B","ANLN","BAG1","BCL2","BIRC5","BLVRA","CCNB1","CCNE1","CDC20","CDC6","CDH3","CENPF",
         "CEP55","CXXC5","EGFR","ERBB2","ESR1","EXO1","FGFR4","FOXA1","FOXC1","GPR160","GRB7","KIF2C",
         "KRT14","KRT17","KRT5","MAPT","MDM2","MELK","MIA","MKI67","MLPH","MMP11","MYBL2","MYC","NAT1",
         "NDC80","NUF2","ORC6","PGR","PHGDH","PTTG1","RRM2","SFRP1","SLC39A6","TMEM45B","TYMS","UBE2C","UBE2T"]

def harmonise(df):
    df = df.rename(columns=lambda g: ALIAS.get(str(g).strip(), str(g).strip()))
    if df.columns.duplicated().any():
        df = df.T.groupby(level=0).mean().T
    return df

def make_xgb(seed=SEED, gpu=False):
    kw = dict(objective="multi:softprob", n_estimators=300, max_depth=5, learning_rate=0.05,
              eval_metric="mlogloss", random_state=seed, n_jobs=-1)
    if gpu:
        kw.update(device="cuda", tree_method="hist")
    return XGBClassifier(**kw)

def make_models(seed=SEED, gpu=False, balanced=False):
    cw = "balanced" if balanced else None
    return {
        "XGBoost": make_xgb(seed, gpu),
        "Random Forest": RandomForestClassifier(n_estimators=300, random_state=seed, n_jobs=-1, class_weight=cw),
        "SVM": SVC(kernel="rbf", C=1.0, gamma="scale", probability=True, random_state=seed, class_weight=cw),
        "Logistic Regression": LogisticRegression(solver="lbfgs", max_iter=5000, class_weight=cw),
    }
MODELS = list(make_models())

def fit_model(name, model, X, y, balanced=False):
    if name == "XGBoost" and balanced:
        model.fit(X, y, sample_weight=compute_sample_weight("balanced", y))
    else:
        model.fit(X, y)
    return model

def tree_shap_importance(booster_model, X):
    c = np.asarray(booster_model.get_booster().predict(xgb.DMatrix(np.asarray(X)), pred_contribs=True))
    if c.ndim == 2:
        c = c.reshape(X.shape[0], -1, X.shape[1] + 1)
    return np.abs(c[:, :, :-1]).mean(axis=(0, 1))

def gain_importance(booster_model, n_features):
    score = booster_model.get_booster().get_score(importance_type="gain")
    g = np.zeros(n_features)
    for key, val in score.items():
        g[int(key[1:])] = val
    return g

def top_k(primary, k, tiebreak=None):
    # rank by primary importance (descending); ties broken by tiebreak (descending), then by column order
    primary = np.nan_to_num(np.asarray(primary, float))
    if tiebreak is None:
        return np.argsort(-primary, kind="stable")[:k]
    tiebreak = np.nan_to_num(np.asarray(tiebreak, float))
    return np.lexsort((-tiebreak, -primary))[:k]

def l1_importance(X, y):
    ovr = OneVsRestClassifier(LogisticRegression(penalty="l1", C=0.5, solver="liblinear", max_iter=2000))
    ovr.fit(X, y)
    return np.mean([np.abs(est.coef_.ravel()) for est in ovr.estimators_], axis=0)

def scores(y, pred, proba=None):
    r = {"Accuracy (%)": accuracy_score(y, pred) * 100,
         "Balanced accuracy (%)": balanced_accuracy_score(y, pred) * 100,
         "Macro F1 (%)": f1_score(y, pred, average="macro") * 100,
         "Weighted F1 (%)": f1_score(y, pred, average="weighted") * 100}
    if proba is not None:
        try:
            r["Weighted ROC-AUC (%)"] = roc_auc_score(y, proba, multi_class="ovr", average="weighted", labels=[0, 1, 2, 3]) * 100
        except ValueError:
            r["Weighted ROC-AUC (%)"] = np.nan
    return r

def bootstrap_acc_ci(y, p, n_boot=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed); n = len(y); ok = (y == p).astype(float)
    b = np.array([ok[rng.integers(0, n, n)].mean() for _ in range(n_boot)]) * 100
    lo, hi = np.percentile(b, [2.5, 97.5])
    return f"{lo:.2f}-{hi:.2f}"

def mcnemar_exact(y, p1, p2):
    a = int(((p1 == y) & (p2 != y)).sum()); b = int(((p1 != y) & (p2 == y)).sum())
    return 1.0 if a + b == 0 else binomtest(a, a + b, 0.5).pvalue

def holm(pvals):
    p = np.asarray(pvals, float); m = len(p); order = np.argsort(p)
    adj = np.empty(m); running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (m - rank) * p[i]); adj[i] = min(1.0, running)
    return adj

def corrected_t(d, n_te_over_n_tr=0.25):
    d = np.asarray(d, float); N = len(d)
    v = d.var(ddof=1)
    if v == 0:
        return 0.0, 1.0
    t = d.mean() / np.sqrt((1 / N + n_te_over_n_tr) * v)
    return t, 2 * stats.t.sf(abs(t), N - 1)

In [ ]:
# ===== 1. Load TCGA-BRCA and METABRIC (same steps as the validation notebook) =====
if not os.path.exists(TCGA_EXPR) and os.path.exists(TCGA_EXPR + ".gz"):
    TCGA_EXPR += ".gz"
if not os.path.exists(TCGA_CLIN):
    alt = glob.glob(f"{CONTENT}/*BRCA_clinicalMatrix*")
    assert alt, "TCGA clinical matrix not found in /content"
    TCGA_CLIN = alt[0]

expr = pd.read_csv(TCGA_EXPR, sep="\t", index_col=0).T
expr.index = expr.index.str[:15]
expr = expr[~expr.index.duplicated(keep="first")]
expr = harmonise(expr.apply(pd.to_numeric, errors="coerce"))
clin = pd.read_csv(TCGA_CLIN, sep="\t", low_memory=False)
id_col = next(c for c in ["sampleID", "sample", "Sample ID", "SAMPLE_ID"] if c in clin.columns)
clin = clin.set_index(id_col); clin.index = clin.index.astype(str).str[:15]
clin = clin[~clin.index.duplicated(keep="first")]

NAME_FIX = {"Luminal A": "LumA", "Luminal B": "LumB", "Basal-like": "Basal", "HER2-enriched": "Her2",
            "Normal-like": "Normal", "LumA": "LumA", "LumB": "LumB", "Basal": "Basal", "Her2": "Her2", "Normal": "Normal"}
def labels_from(col):
    lab = clin[col].dropna().astype(str).str.strip().map(NAME_FIX)
    return lab[lab.isin(CLASSES)]

lab_t = labels_from("PAM50_mRNA_nature2012")
ids_t = expr.index.intersection(lab_t.index)
X_tcga = expr.loc[ids_t]
if X_tcga.isna().any().any():
    X_tcga = X_tcga.fillna(X_tcga.median())
y_tcga = lab_t.loc[ids_t].map(LABEL_MAP).astype(int).values
print("TCGA tumours (PAM50_mRNA_nature2012, four classes):", len(ids_t), "(paper: 514)")

# --- Check: would PAM50Call_RNAseq give a larger cohort?
if "PAM50Call_RNAseq" in clin.columns:
    lab_rs = labels_from("PAM50Call_RNAseq")
    ids_rs = expr.index.intersection(lab_rs.index)
    both = ids_t.intersection(ids_rs)
    agree = (lab_rs.loc[both] == lab_t.loc[both]).mean() * 100 if len(both) else np.nan
    print(f"PAM50Call_RNAseq: {len(ids_rs)} tumours with expression and a four-class label; "
          f"{len(both)} overlap with the 514; label agreement on overlap = {agree:.1f}%")
    print(lab_rs.loc[ids_rs].value_counts().to_dict())
    SUMMARY["PAM50Call_RNAseq tumours (4 classes)"] = len(ids_rs)
    SUMMARY["PAM50Call_RNAseq agreement with nature2012 (%)"] = round(float(agree), 1)
else:
    print("PAM50Call_RNAseq column not present in this clinical file")

# --- METABRIC
if not os.path.exists(os.path.join(METABRIC_DIR, "data_mrna_illumina_microarray.txt")):
    tars = sorted(glob.glob(f"{CONTENT}/brca_metabric*.tar.gz"), key=os.path.getsize, reverse=True)
    assert tars, "Upload brca_metabric.tar.gz (or the extracted folder) to /content first"
    with tarfile.open(tars[0]) as t:
        t.extractall(CONTENT)
mb = pd.read_csv(os.path.join(METABRIC_DIR, "data_mrna_illumina_microarray.txt"), sep="\t", comment="#", low_memory=False)
mb = mb.dropna(subset=["Hugo_Symbol"]).drop(columns=[c for c in ["Entrez_Gene_Id"] if c in mb.columns])
mb = harmonise(mb.set_index("Hugo_Symbol").T.apply(pd.to_numeric, errors="coerce"))
pat = pd.read_csv(os.path.join(METABRIC_DIR, "data_clinical_patient.txt"), sep="\t", comment="#", index_col="PATIENT_ID")
smp = pd.read_csv(os.path.join(METABRIC_DIR, "data_clinical_sample.txt"), sep="\t", comment="#")
s2p = dict(zip(smp["SAMPLE_ID"], smp["PATIENT_ID"]))
lab_m_all = pd.Series({s: pat.loc[s2p[s], "CLAUDIN_SUBTYPE"] if s2p.get(s) in pat.index else np.nan for s in mb.index})
lab_m = lab_m_all[lab_m_all.isin(CLASSES)]
X_mb = mb.loc[lab_m.index]
miss = X_mb.isna().mean(); X_mb = X_mb.loc[:, miss <= 0.20]; X_mb = X_mb.fillna(X_mb.median())
y_mb = lab_m.map(LABEL_MAP).astype(int).values
print("METABRIC tumours (four classes):", len(y_mb), "(previous run: 1608)")

common = sorted(set(X_tcga.columns) & set(X_mb.columns))
pam50_common = [g for g in PAM50 if g in common]
Xt_all = pd.DataFrame(StandardScaler().fit_transform(X_tcga.values), index=X_tcga.index, columns=X_tcga.columns)
Xt = Xt_all[common]
Xm = pd.DataFrame(StandardScaler().fit_transform(X_mb[common].values), index=X_mb.index, columns=common)
print(f"Common genes: {len(common)} (previous run: 16907) | PAM50 in both: {len(pam50_common)}/50")
majority = pd.Series(y_mb).value_counts().max() / len(y_mb) * 100

In [ ]:
# ===== Preparation for Section 7: SHAP-300 panel and SVM predictions on METABRIC =====
path = f"{OUT_DIR}/imp_shap_gain_common.npy"
if os.path.exists(path):
    imp_common = np.load(path)
else:
    print("Fitting the selection booster (about 15 min on CPU)...")
    b = make_xgb().fit(Xt.values, y_tcga)
    imp_common = np.vstack([tree_shap_importance(b, Xt.values), gain_importance(b, Xt.shape[1])])
    np.save(path, imp_common)
shap300 = list(np.array(common)[top_k(imp_common[0], PANEL_SIZE)])
panels_ext = {"SHAP-300": shap300}
svm = make_models()["SVM"].fit(Xt[shap300].values, y_tcga)
preds_ext = {("SHAP-300", "SVM"): svm.predict(Xm[shap300].values)}
print("SVM METABRIC accuracy:", round(accuracy_score(y_mb, preds_ext[("SHAP-300", "SVM")]) * 100, 2), "(should be 81.28)")

In [ ]:
# ===== 7. Survival in METABRIC: do the predicted subtypes separate overall survival like the true ones? =====
def km_curve(t, e):
    order = np.argsort(t); t, e = t[order], e[order]
    times = np.unique(t[e == 1]); s, surv = 1.0, []
    for u in times:
        at_risk = (t >= u).sum(); d = ((t == u) & (e == 1)).sum(); s *= 1 - d / at_risk; surv.append(s)
    return np.concatenate([[0], times]), np.concatenate([[1.0], surv])

def logrank(t, e, g):
    groups = np.unique(g); times = np.unique(t[e == 1]); K = len(groups)
    O = np.zeros(K); E = np.zeros(K); V = np.zeros((K, K))
    for u in times:
        at = t >= u; n = at.sum(); d = ((t == u) & (e == 1)).sum()
        nj = np.array([(at & (g == k)).sum() for k in groups]); dj = np.array([((t == u) & (e == 1) & (g == k)).sum() for k in groups])
        O += dj; E += d * nj / n
        if n > 1:
            V += d * (n - d) / (n - 1) * (np.diag(nj / n) - np.outer(nj, nj) / n ** 2)
    diff = (O - E)[:-1]; chi2 = float(diff @ np.linalg.pinv(V[:-1, :-1]) @ diff)
    return chi2, stats.chi2.sf(chi2, K - 1)

pid = [s2p.get(s) for s in X_mb.index]
os_m = pd.to_numeric(pat.reindex(pid)["OS_MONTHS"], errors="coerce").values
os_e = pat.reindex(pid)["OS_STATUS"].astype(str).str.startswith("1").values.astype(int)
ok = ~np.isnan(os_m)
pred_svm = preds_ext[("SHAP-300", "SVM")]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
res7 = {}
for ax, (title, lab) in zip(axes, [("a  METABRIC PAM50 labels", y_mb), ("b  Predicted by SHAP-300 + SVM (trained on TCGA)", pred_svm)]):
    for k, name in enumerate(CLASS_NAMES):
        m = ok & (lab == k)
        if m.sum() > 0:
            x_, y_ = km_curve(os_m[m], os_e[m]); ax.step(x_ / 12, y_, where="post", label=f"{name} (n={m.sum()})")
    chi2, p = logrank(os_m[ok], os_e[ok], lab[ok]); res7[title[:1]] = (round(chi2, 2), p)
    ax.set_title(title, loc="left", fontsize=11); ax.set_xlabel("Years"); ax.text(0.02, 0.05, f"log-rank p = {p:.1e}", transform=ax.transAxes)
    ax.legend(frameon=False, fontsize=8, loc="lower right"); ax.set_xlim(0, 25)
axes[0].set_ylabel("Overall survival probability")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_metabric_survival.png", dpi=600); plt.show()
print("log-rank chi2 and p (a = true labels, b = predicted):", res7)
SUMMARY["Sec7 log-rank (true labels)"] = res7["a"]; SUMMARY["Sec7 log-rank (predicted)"] = res7["b"]

In [ ]:
# ===== 8. Pathway enrichment of the SHAP-300 panel (Enrichr; needs internet) =====
try:
    try:
        import gseapy
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gseapy"]); import gseapy
    enr = gseapy.enrichr(gene_list=panels_ext["SHAP-300"], organism="human", outdir=None,
                         gene_sets=["MSigDB_Hallmark_2020", "KEGG_2021_Human", "GO_Biological_Process_2023"])
    er = enr.results.sort_values("Adjusted P-value")
    sig = er[er["Adjusted P-value"] < 0.05]
    display(sig.groupby("Gene_set").head(8)[["Gene_set", "Term", "Overlap", "Adjusted P-value", "Genes"]])
    er.to_csv(f"{OUT_DIR}/sec8_enrichment.csv", index=False)
    SUMMARY["Sec8 enrichment (top, adj p<0.05)"] = sig.groupby("Gene_set").head(6)[["Gene_set", "Term", "Overlap", "Adjusted P-value"]].to_dict("records")
except Exception as e:
    print("Enrichment failed (check internet):", e)

In [ ]:
# ===== Fast settings for Section 6 (5 outer folds instead of 25) =====
# The full 25-fold version needs about 13 h on CPU, which is longer than a Colab session.
# The panel-size sweep and the tuning are sensitivity analyses, so the first repeat (seed 42, 5 folds) is enough.
N_REPEATS = 1
RUN_NOPAM50_IN_CV = False
RUN_ALLGENE_XGB_CV = False
USE_GPU_CV = gpu_available()
print("GPU for Section 6:", USE_GPU_CV, "| expected time:", "about 20-30 min" if USE_GPU_CV else "about 1.5-2 h")
old = f"{OUT_DIR}/sec6_nested_cv_rows.csv"
if os.path.exists(old):
    shutil.move(old, f"{OUT_DIR}/sec6_old_partial_run.csv")
    print("An older partial file was moved aside; Section 6 starts fresh.")

In [ ]:
# ===== 6. Nested CV: panel-size sweep, tuned SVM / LR, and tests on balanced accuracy and macro F1 =====
# Same protocol as the paper: 5 repeats x 5 stratified folds (seeds 42-46); scaling and every selection rule
# are fitted on the training fold only. Results are appended to a CSV after each fold, so the cell can be re-run.
CV_CSV = f"{OUT_DIR}/sec6_nested_cv_rows.csv"
done = set()
if os.path.exists(CV_CSV):
    _prev = pd.read_csv(CV_CSV); done = set(zip(_prev["repeat"], _prev["fold"]))
    print("Already finished folds:", len(done))
X_all_np = X_tcga.values; all_cols = np.array(X_tcga.columns)
pam_idx = np.array([i for i, g in enumerate(all_cols) if g in PAM50])
nopam_idx = np.array([i for i, g in enumerate(all_cols) if g not in PAM50])
SVM_GRID = {"C": [0.1, 1, 10], "gamma": ["scale", 1e-4, 1e-3]}
LR_GRID = {"C": [0.01, 0.1, 1, 10]}

def eval_set(tag, k, Xtr, ytr, Xte, yte, r, f, gpu, which=MODELS):
    out = []
    for mname, model in make_models(SEED + r, gpu).items():
        if mname not in which:
            continue
        fit_model(mname, model, Xtr, ytr); p = model.predict(Xte)
        out.append({"repeat": r, "fold": f, "Feature set": tag, "k": k, "Model": mname, **scores(yte, p)})
    return out

t_start = time.time()
for r in range(N_REPEATS):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED + r)
    for f, (tr_i, te_i) in enumerate(skf.split(X_all_np, y_tcga)):
        if (r, f) in done:
            continue
        t0 = time.time(); fold_rows = []
        sc = StandardScaler().fit(X_all_np[tr_i]); Xtr = sc.transform(X_all_np[tr_i]); Xte = sc.transform(X_all_np[te_i])
        ytr, yte = y_tcga[tr_i], y_tcga[te_i]
        b = make_xgb(SEED + r, USE_GPU_CV).fit(Xtr, ytr)
        shap_imp = tree_shap_importance(b, Xtr); gain_imp = gain_importance(b, Xtr.shape[1])
        F, _ = f_classif(Xtr, ytr)
        order = top_k(shap_imp, max(K_LIST))
        for k in K_LIST:                                            # panel-size sweep
            g = order[:k]
            fold_rows += eval_set("SHAP", k, Xtr[:, g], ytr, Xte[:, g], yte, r, f, USE_GPU_CV)
        g = order[:PANEL_SIZE]                                       # tuned SVM and LR at k = 300
        inner = StratifiedKFold(3, shuffle=True, random_state=SEED + r)
        for mname, est, grid in [("SVM (tuned)", SVC(kernel="rbf", probability=False, random_state=SEED), SVM_GRID),
                                 ("Logistic Regression (tuned)", LogisticRegression(solver="lbfgs", max_iter=5000), LR_GRID)]:
            gs = GridSearchCV(est, grid, cv=inner, scoring="accuracy", n_jobs=-1).fit(Xtr[:, g], ytr)
            p = gs.predict(Xte[:, g])
            fold_rows.append({"repeat": r, "fold": f, "Feature set": "SHAP", "k": PANEL_SIZE, "Model": mname,
                              **scores(yte, p), "best params": json.dumps(gs.best_params_)})
        comps = {"Gain": top_k(gain_imp, PANEL_SIZE, F), "ANOVA": top_k(F, PANEL_SIZE),
                 "L1": top_k(l1_importance(Xtr, ytr), PANEL_SIZE, F),
                 "Random": np.random.default_rng(SEED + 100 * r + f).choice(Xtr.shape[1], PANEL_SIZE, replace=False),
                 "PAM50": pam_idx}
        if RUN_NOPAM50_IN_CV:
            b2 = make_xgb(SEED + r, USE_GPU_CV).fit(Xtr[:, nopam_idx], ytr)
            comps["SHAP without PAM50"] = nopam_idx[top_k(tree_shap_importance(b2, Xtr[:, nopam_idx]), PANEL_SIZE)]
        for tag, g in comps.items():
            fold_rows += eval_set(tag, len(g), Xtr[:, g], ytr, Xte[:, g], yte, r, f, USE_GPU_CV)
        allm = MODELS if RUN_ALLGENE_XGB_CV else [m for m in MODELS if m != "XGBoost"]
        fold_rows += eval_set("All genes", Xtr.shape[1], Xtr, ytr, Xte, yte, r, f, USE_GPU_CV, which=allm)
        pd.DataFrame(fold_rows).to_csv(CV_CSV, mode="a", header=not os.path.exists(CV_CSV), index=False)
        print(f"repeat {r} fold {f} done in {(time.time() - t0) / 60:.1f} min (total {(time.time() - t_start) / 60:.1f} min)")

cv = pd.read_csv(CV_CSV)
n_folds = cv[["repeat", "fold"]].drop_duplicates().shape[0]
print("Outer folds finished:", n_folds)

In [ ]:
# ===== 6b. Results of Section 6: tables and the panel-size figure =====
metrics = ["Accuracy (%)", "Balanced accuracy (%)", "Macro F1 (%)"]
sweep = cv[(cv["Feature set"] == "SHAP") & cv["Model"].isin(MODELS)]
sweep_tab = sweep.groupby(["k", "Model"])[metrics].agg(["mean", "std"]).round(2)
display(sweep_tab)
sweep_tab.to_csv(f"{OUT_DIR}/sec6_panel_size_sweep.csv")

fig, ax = plt.subplots(figsize=(7, 4.5))
for mname, mk in zip(MODELS, ["o", "s", "^", "D"]):
    s = sweep[sweep["Model"] == mname].groupby("k")["Accuracy (%)"].agg(["mean", "std"])
    ax.errorbar(s.index, s["mean"], yerr=s["std"], marker=mk, capsize=3, label=mname)
ax.set_xscale("log"); ax.set_xticks(K_LIST); ax.set_xticklabels(K_LIST)
ax.set_xlabel("Panel size k (genes)"); ax.set_ylabel("Nested CV accuracy (%)")
ax.legend(frameon=False, fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_panel_size_sweep.png", dpi=600); plt.show()

tuned = cv[cv["Model"].str.contains("tuned")]
fixed = cv[(cv["Feature set"] == "SHAP") & (cv["k"] == PANEL_SIZE) & cv["Model"].isin(["SVM", "Logistic Regression"])]
tune_tab = pd.concat([fixed, tuned]).groupby("Model")[metrics].agg(["mean", "std"]).round(2)
display(tune_tab)
print("Most frequent tuned settings:")
print(tuned.groupby("Model")["best params"].agg(lambda s: s.value_counts().index[0]))
tune_tab.to_csv(f"{OUT_DIR}/sec6_tuning.csv")

# corrected resampled t-test, SHAP-300 minus each comparator, for all three metrics (Holm within classifier and metric)
base = cv[(cv["Feature set"] == "SHAP") & (cv["k"] == PANEL_SIZE) & cv["Model"].isin(MODELS)]
comp_names = [c for c in ["Gain", "ANOVA", "L1", "Random", "PAM50", "SHAP without PAM50", "All genes"] if c in set(cv["Feature set"])]
test_rows = []
for metric in metrics:
    for mname in MODELS:
        b = base[base["Model"] == mname].set_index(["repeat", "fold"])[metric]
        raw, diffs, names = [], [], []
        for c in comp_names:
            o = cv[(cv["Feature set"] == c) & (cv["Model"] == mname)].set_index(["repeat", "fold"])[metric]
            if len(o) == 0:
                continue
            d = (b - o).dropna(); t, p = corrected_t(d.values)
            raw.append(p); diffs.append(d.mean()); names.append(c)
        for c, dm, p, pa in zip(names, diffs, raw, holm(raw)):
            test_rows.append({"Metric": metric, "Model": mname, "SHAP-300 minus": c,
                              "Mean difference (points)": round(dm, 2), "p": round(p, 4), "Holm-adjusted p": round(pa, 4)})
tests = pd.DataFrame(test_rows); display(tests)
tests.to_csv(f"{OUT_DIR}/sec6_corrected_t_tests_all_metrics.csv", index=False)
acc_check = cv[cv["Model"].isin(MODELS) & ((cv["k"] == PANEL_SIZE) | (cv["Feature set"] != "SHAP"))]
print("Reproduction check against paper Table 6 (accuracy means):")
display(acc_check.groupby(["Feature set", "Model"])["Accuracy (%)"].mean().unstack().round(2))
SUMMARY["Sec6 outer folds"] = int(n_folds)
SUMMARY["Sec6 sweep accuracy mean"] = sweep.groupby(["k", "Model"])["Accuracy (%)"].mean().unstack().round(2).to_dict()
SUMMARY["Sec6 sweep accuracy sd"] = sweep.groupby(["k", "Model"])["Accuracy (%)"].std().unstack().round(2).to_dict()
SUMMARY["Sec6 tuning"] = pd.concat([fixed, tuned]).groupby("Model")[metrics].mean().round(2).to_dict()
SUMMARY["Sec6 tuned best params"] = tuned.groupby("Model")["best params"].agg(lambda s: s.value_counts().index[0]).to_dict()
SUMMARY["Sec6 significant tests (Holm p<0.05)"] = tests[tests["Holm-adjusted p"] < 0.05].to_dict("records")
SUMMARY["Sec6 Table6 reproduction"] = acc_check.groupby(["Feature set", "Model"])["Accuracy (%)"].mean().unstack().round(2).to_dict()